# 02 – Mô hình


In [1]:
# Colab: Runtime > Change runtime type > GPU (T4)
from google.colab import drive
drive.mount('/content/drive')
import sys
PROJECT = "/content/drive/MyDrive/FER2013_PyTorch"
sys.path.insert(0, PROJECT)
from common import *
print("Device:", DEV, "| torch", torch.__version__)

Mounted at /content/drive
Device: cuda | torch 2.14.1+cu130


## 2.1 Ghi module `models.py`

In [2]:
%%writefile /content/drive/MyDrive/FER2013_PyTorch/models.py
"""Kiến trúc CNN (Tiến – 3123580052). ReLU không in-place để Grad-CAM hoạt động."""
from common import *

CFG = {"cnn_small":  dict(blocks=[(64, 2), (128, 2), (256, 2)], fc=1024),
       "cnn_medium": dict(blocks=[(64, 3), (128, 4), (256, 4)], fc=1024),
       "cnn_large":  dict(blocks=[(64, 4), (128, 4), (256, 4), (512, 4)], fc=2048)}

class VGGLike(nn.Module):
    def __init__(self, blocks, fc, num_classes=7, in_size=48, bn=True, conv_drop=0.2, fc_drop=0.5):
        super().__init__()
        layers, cin = [], 1
        for cout, n in blocks:
            for _ in range(n):
                layers.append(nn.Conv2d(cin, cout, 3, padding=1, bias=not bn))
                if bn: layers.append(nn.BatchNorm2d(cout))
                layers.append(nn.ReLU(inplace=False)); cin = cout
            layers.append(nn.MaxPool2d(2))
            if conv_drop > 0: layers.append(nn.Dropout(conv_drop))
        self.features = nn.Sequential(*layers)
        with torch.no_grad(): feat = self.features(torch.zeros(1, 1, in_size, in_size)).numel()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(feat, fc), nn.ReLU(inplace=False),
                                        nn.Dropout(fc_drop), nn.Linear(fc, num_classes))
    def forward(self, x): return self.classifier(self.features(x))

class VGG16FT(nn.Module):
    def __init__(self, num_classes=7, pretrained=True, fc_drop=0.5):
        super().__init__()
        from torchvision.models import vgg16
        m = vgg16(weights="IMAGENET1K_V1" if pretrained else None)
        for mod in m.modules():
            if isinstance(mod, nn.ReLU): mod.inplace = False
        self.features = m.features
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, 2048),
                                        nn.ReLU(inplace=False), nn.Dropout(fc_drop), nn.Linear(2048, num_classes))
    def forward(self, x): return self.classifier(self.features(x.repeat(1, 3, 1, 1)))

def build_model(name, img_size=48, bn=True, conv_drop=0.2, fc_drop=0.5, pretrained=True):
    if name == "vgg16_ft": return VGG16FT(7, pretrained, fc_drop)
    return VGGLike(CFG[name]["blocks"], CFG[name]["fc"], 7, img_size, bn, conv_drop, fc_drop)

def last_conv(model): return [m for m in model.modules() if isinstance(m, nn.Conv2d)][-1]
def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

Writing /content/drive/MyDrive/FER2013_PyTorch/models.py


## 2.2 Số tham số và kiểm tra kích thước

In [3]:
from models import *
for n in CFG:
    m = build_model(n)
    print(f'{n:11s} params = {count_params(m):,} | out shape =', tuple(m(torch.zeros(2,1,48,48)).shape))
m = build_model('vgg16_ft', pretrained=False)
print(f'{"vgg16_ft":11s} params = {count_params(m):,} | out shape =', tuple(m(torch.zeros(2,1,64,64)).shape))

cnn_small   params = 10,590,535 | out shape = (2, 7)
cnn_medium  params = 12,103,623 | out shape = (2, 7)
cnn_large   params = 20,410,439 | out shape = (2, 7)
vgg16_ft    params = 15,779,655 | out shape = (2, 7)


## 2.3 Cấu trúc baseline (`cnn_medium`)

In [4]:
print(build_model('cnn_medium'))

VGGLike(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (7): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Dropout(p=0.2, inplace=False)
    (11): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (12): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (13): ReLU()
    (14): Conv2d(128, 128